# Multi-turn conversations

The API is stateless: it does not remember earlier requests. To continue a conversation,
keep the history yourself and send all of it every time.

- store every user and assistant message in one `messages` list
- after each reply, append it as an `assistant` message
- the next request sends the full list, so Claude sees the context

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return "".join(block.text for block in message.content if block.type == "text")

## A two-turn conversation

In [3]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)
print("Answer 1:", answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print("Answer 2:", final_answer)

Answer 1: Quantum computing harnesses quantum mechanics principles like superposition and entanglement to process information in fundamentally different ways than classical computers, enabling faster solutions to certain complex problems.


Answer 2: Unlike classical bits that are either 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously, allowing quantum computers to explore many possible solutions in parallel.


"Write another sentence" only makes sense with the history: the second answer continues
the topic of the first one.